In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH10-Retriever")

loader = PyMuPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()


C:\Users\user\AppData\Local\Temp\ipykernel_5516\3915805956.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
d:\rag_one\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever


In [2]:
print(docs[5].page_content[:500])

1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
영국 AI 안전성 정상회의에 참가한 28개국, AI 위험에 공동 대응 선언
n 영국 블레츨리 파크에서 개최된 AI 안전성 정상회의에 참가한 28개국들이 AI 안전 보장을 
위한 협력 방안을 담은 블레츨리 선언을 발표
n 첨단 AI를 개발하는 국가와 기업들은 AI 시스템에 대한 안전 테스트 계획에 합의했으며, 
영국의 AI 안전 연구소가 전 세계 국가와 협력해 테스트를 주도할 예정 
KEY Contents
£ AI 안전성 정상회의 참가국들, 블레츨리 선언 통해 AI 안전 보장을 위한 협력에 합의
n 2023년 11월 1~2일 영국 블레츨리 파크에서 열린 AI 안전성 정상회의(AI Safety Summit)에 
참가한 28개국 대표들이 AI 위험 관리를 위한 ‘블레츨리 선언’을 발표 
∙선언은 AI 안전 보장을 위해 국가, 국제기구, 기업, 시민사회, 학계를 포함한 모든 이해관계자의 협력이 
중요하다고 강조했으며,


In [3]:
import uuid
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

vectorstore = Chroma(
    collection_name="samll_bigger_chunks",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)

store = InMemoryStore() #부모 문서의 저장소 계층

id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore = vectorstore,
    byte_store=store,
    id_key=id_key,
)

doc_ids = [str(uuid.uuid4()) for _ in docs]
doc_ids


['06b3f12d-3841-432c-9d32-cae58b79de4c',
 '4442bef9-c7e9-4321-a40c-afdfeed2c3e1',
 '9ddf6a1f-f7ea-4379-8e9e-e1a7b9f0e020',
 '55c02c08-686a-4cbd-9a88-cfd43542c002',
 'ab25919b-2ddc-45e6-8ff0-2a9dcffc5652',
 '68073e90-a50b-4ea2-9e74-e056586f5225',
 'cb0d5027-a7e0-485a-be25-d718c97826a0',
 'b246cd5e-64a5-4b63-95f6-ee534fc32078',
 '2cc42ebc-5010-4215-82d3-1bd3416195ba',
 '3d103e0b-62e6-4afc-8f78-caa49dbdbb36',
 '9086f165-353e-43b0-9dde-74af41e9d024',
 '38abaed7-8e30-4408-b91f-cc377530e02f',
 '886a00c6-fa14-4c58-bc87-0c63a7e12716',
 '13a0b61d-49b9-4c54-92de-10d7d13f71f6',
 '62f582fa-2349-4233-8557-63e9dd3f9a3a',
 'b0ec763b-58c6-468b-8fed-dd1e22f078fb',
 '979b8337-bff8-4ab7-a9a8-860096cc84fc',
 '0fc61fdc-60dd-437e-a7c7-eb9ed368c8d2',
 '5585eddb-070b-4cd9-9901-8c646d456265',
 '239747ab-6bf4-425e-837f-9226b1e1beeb',
 'e5e8d84c-49cc-4d01-9e3c-95dce400d8bf',
 '709bda48-20ce-4260-bd4c-8954b3b56f11',
 '83b141f2-fbe2-4e3f-8fef-3789219b26bb']

In [12]:
parent_text_splitter = RecursiveCharacterTextSplitter(chunk_size=600)

child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

parent_docs = []
for i, doc in enumerate(docs):
    print(f"{i} = {doc_ids[i]}")
    _id = doc_ids[i]
    parent_doc =parent_text_splitter.split_documents([doc])
    
    for _doc in parent_doc:
        _doc.metadata[id_key] = _id
    parent_docs.extend(parent_doc)
    print(f"{i} extend {parent_docs[i].metadata[id_key]}")

0 = 06b3f12d-3841-432c-9d32-cae58b79de4c
0 extend 06b3f12d-3841-432c-9d32-cae58b79de4c
1 = 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
1 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
2 = 9ddf6a1f-f7ea-4379-8e9e-e1a7b9f0e020
2 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
3 = 55c02c08-686a-4cbd-9a88-cfd43542c002
3 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
4 = ab25919b-2ddc-45e6-8ff0-2a9dcffc5652
4 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
5 = 68073e90-a50b-4ea2-9e74-e056586f5225
5 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
6 = cb0d5027-a7e0-485a-be25-d718c97826a0
6 extend 9ddf6a1f-f7ea-4379-8e9e-e1a7b9f0e020
7 = b246cd5e-64a5-4b63-95f6-ee534fc32078
7 extend 55c02c08-686a-4cbd-9a88-cfd43542c002
8 = 2cc42ebc-5010-4215-82d3-1bd3416195ba
8 extend 55c02c08-686a-4cbd-9a88-cfd43542c002
9 = 3d103e0b-62e6-4afc-8f78-caa49dbdbb36
9 extend 55c02c08-686a-4cbd-9a88-cfd43542c002
10 = 9086f165-353e-43b0-9dde-74af41e9d024
10 extend 55c02c08-686a-4cbd-9a88-cfd43542c002
11 = 38abaed7-8e30-4408-b91f-cc377530e02f

In [18]:
parent_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0,
 'doc_id': '06b3f12d-3841-432c-9d32-cae58b79de4c'}

In [14]:
child_docs = []
for i, doc in enumerate(docs):
    print(f"{i} = {doc_ids[i]}")
    _id = doc_ids[i]   
    child_doc = child_text_splitter.split_documents([doc])
    
    for _doc in child_doc:
        _doc.metadata[id_key] = _id
        
    child_docs.extend(child_doc)
    print(f"{i} extend {child_docs[i].metadata[id_key]}")
    

0 = 06b3f12d-3841-432c-9d32-cae58b79de4c
0 extend 06b3f12d-3841-432c-9d32-cae58b79de4c
1 = 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
1 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
2 = 9ddf6a1f-f7ea-4379-8e9e-e1a7b9f0e020
2 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
3 = 55c02c08-686a-4cbd-9a88-cfd43542c002
3 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
4 = ab25919b-2ddc-45e6-8ff0-2a9dcffc5652
4 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
5 = 68073e90-a50b-4ea2-9e74-e056586f5225
5 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
6 = cb0d5027-a7e0-485a-be25-d718c97826a0
6 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
7 = b246cd5e-64a5-4b63-95f6-ee534fc32078
7 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
8 = 2cc42ebc-5010-4215-82d3-1bd3416195ba
8 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
9 = 3d103e0b-62e6-4afc-8f78-caa49dbdbb36
9 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
10 = 9086f165-353e-43b0-9dde-74af41e9d024
10 extend 4442bef9-c7e9-4321-a40c-afdfeed2c3e1
11 = 38abaed7-8e30-4408-b91f-cc377530e02f

In [15]:
child_docs[0].metadata

{'producer': 'Hancom PDF 1.3.0.542',
 'creator': 'Hwp 2018 10.0.0.13462',
 'creationdate': '2023-12-08T13:28:38+09:00',
 'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'total_pages': 23,
 'format': 'PDF 1.4',
 'title': '',
 'author': 'dj',
 'subject': '',
 'keywords': '',
 'moddate': '2023-12-08T13:28:38+09:00',
 'trapped': '',
 'modDate': "D:20231208132838+09'00'",
 'creationDate': "D:20231208132838+09'00'",
 'page': 0,
 'doc_id': '06b3f12d-3841-432c-9d32-cae58b79de4c'}

In [19]:
print(f"분할된 parent_docs의 개수 : {len(parent_docs)}")
print(f"분할된 child_docs의 개수 : {len(child_docs)}")

분할된 parent_docs의 개수 : 73
분할된 child_docs의 개수 : 440


In [20]:
retriever.vectorstore.add_documents(parent_docs)
retriever.vectorstore.add_documents(child_docs)


retriever.docstore.mset(list(zip(doc_ids, docs)))

In [21]:
relevant_chunks = retriever.vectorstore.similarity_search(
    "삼성전자가 만든 생성형 AI의 이름은?"
)

print(f"검색된 문서의 개수: {len(relevant_chunks)}")

검색된 문서의 개수: 4


In [22]:
for chunk in relevant_chunks:
    print(chunk.page_content, end ="\n\n")
    print(">" * 100, end="\n\n")

☞ 출처 : 삼성전자, ‘삼성 AI 포럼’서 자체 개발 생성형 AI ‘삼성 가우스’ 공개, 2023.11.08.
삼성전자, ‘삼성 개발자 콘퍼런스 코리아 2023’ 개최, 2023.11.14.
TechRepublic, Samsung Gauss: Samsung Research Reveals Generative AI, 2023.11.08.

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

▹ 삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개 ··························································· 10
   ▹ 구글, 앤스로픽에 20억 달러 투자로 생성 AI 협력 강화 ················································ 11

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 

In [23]:
relevant_docs = retriever.invoke("삼성전자가 만든 생성형 AI의 이름은?")
print(f"검색된 문서의 개수: {len(relevant_docs)}", end="\n\n")
print("="*100, end="\n\n")
print(relevant_docs[0].page_content)

검색된 문서의 개수: 2


SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 
이미지 모델의 3개 모델로 구성
∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 
처리를 지원
∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 
사내 소프트웨어 개발에 최적화
∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 
저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 
2024년부터 가우스를 탑재한 삼성

In [25]:
from langchain_classic.retrievers.multi_vector import SearchType

retriever.search_type = SearchType.similarity_score_threshold
retriever.search_kwargs = {"score_threshold":0.3}

print(retriever.invoke("삼성전자가 만든 생성형 AI의 이름은?")[0].page_content)

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에 
단계적으로 탑재할 계획
n 삼성 가우스는 △텍스트를 생성하는 언어모델 △코드를 생성하는 코드 모델 △이미지를 생성하는 
이미지 모델의 3개 모델로 구성
∙언어 모델은 클라우드와 온디바이스 대상 다양한 모델로 구성되며, 메일 작성, 문서 요약, 번역 업무의 
처리를 지원
∙코드 모델 기반의 AI 코딩 어시스턴트 ‘코드아이(code.i)’는 대화형 인터페이스로 서비스를 제공하며 
사내 소프트웨어 개발에 최적화
∙이미지 모델은 창의적인 이미지를 생성하고 기존 이미지를 원하는 대로 바꿀 수 있도록 지원하며 
저해상도 이미지의 고해상도 전환도 지원
n IT 전문지 테크리퍼블릭(TechRepublic)은 온디바이스 AI가 주요 기술 트렌드로 부상했다며, 
2024년부터 가우스를 탑재한 삼성 스마트폰이 메타의 라마(Ll

In [26]:
from langchain_classic.retrievers.multi_vector import SearchType

retriever.search_type = SearchType.similarity
retriever.search_kwargs={"k":1}

print(len(retriever.invoke("삼성전자가 만든 생성형 AI의 이름은?")))

1


In [27]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap = 50)

split_docs = loader.load_and_split(text_splitter)

print(f"분할된 문서의 개수 : {len(split_docs)}")

분할된 문서의 개수 : 61


In [28]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

summary_chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert in summarizing documents in Korean."),
            ("user", "Summarize the following documents in 3 sentences in bullet points format.\n\n{doc}",),
        ]
    )
    | ChatOpenAI(temperature=0,model="gpt-4o-mini")
    | StrOutputParser()
)

In [29]:
summaries = summary_chain.batch(split_docs,{"max_concurrency": 10})

In [30]:
len(summaries)

61

In [31]:
print(split_docs[33].page_content, end="\n\n")
print("[요약]")
print(summaries[33])

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에

[요약]
- 삼성전자가 자체 개발한 생성 AI 모델 '삼성 가우스'를 공개하였으며, 이 모델은 언어, 코드, 이미지의 3개 모델로 구성되어 온디바이스에서 작동 가능하다.  
- '삼성 가우스'는 정규분포 이론을 정립한 수학자 가우스의 이름을 따왔으며, 다양한 상황에 최적화된 모델 선택이 가능하다.  
- 삼성전자는 이 AI 모델이 사용자 정보를 외부로 유출하지 않도록 설계되었으며, 향후 다양한 제품에 단계적으로 탑재할 계획이다.


In [32]:
import uuid
summary_vectorstore = Chroma(
    collection_name="summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
)

store = InMemoryStore() 

id_key = "doc_id"
retriever = MultiVectorRetriever(
    vectorstore=summary_vectorstore,
    byte_store=store,
    id_key=id_key,
)

doc_ids = [str(uuid.uuid4()) for _ in split_docs]

In [33]:
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]}) for i,s in enumerate(summaries)
]
len(summary_docs)

61

In [34]:
retriever.vectorstore.add_documents(
    summary_docs
)

['cb3d144a-735e-4db3-844b-1fee0869702f',
 '6b4664f6-0fc5-4e7b-9f48-6a665ebcbf89',
 '1a3de68f-bf89-462d-a1a1-5f1b94f33b43',
 '95bd77a9-00e4-4ac1-9bc1-2d2712088202',
 'b8a21b3e-fc88-4df6-bdfc-3ffb2b8e1b28',
 '45a90d13-2940-4052-b47c-dccec813813c',
 '6797e5b0-1d72-4969-a5f1-e6750972868f',
 '10d5e575-e275-454b-9029-8eb6c75f8337',
 '4c3aacf6-0919-4863-92ca-4f87e57e874d',
 'f85e2643-1e2c-4415-a58b-3297c8e39665',
 'a068a6ca-e7ae-4036-bc5d-8aa1a794f244',
 '6615441d-ab05-43c6-bea8-eb8c8bded435',
 '268fe36a-210c-4aa5-887c-1a59f35eec97',
 '7ee18596-285a-4953-b135-0f6ec6c8f18a',
 '7a428eb5-8d52-41ba-818a-1675c3a79487',
 '2629942d-4df1-4702-bd73-ecbc2c256a7f',
 '41a9ca0b-88d7-4309-8f74-9dc5595c2285',
 '5ddaaf88-4014-4ab0-986d-62dbabc98bb4',
 'b0e7b901-a464-4070-bf7d-4fb92f293c38',
 '186bfe2d-fb0c-45c2-a37a-d950140a6498',
 'd5b426b8-c6dc-478d-a6bb-d7ba06b80e2d',
 'f455004f-bea5-4ca3-83ea-d3114a1010c8',
 '8141b33e-d4ae-4bc6-bad1-966afd76c2bf',
 '1c18e759-63e9-445f-80d1-79b5fe611b6f',
 '248cbc40-3017-

In [35]:
retriever.docstore.mset(list(zip(doc_ids,split_docs)))

In [36]:
result_docs = summary_vectorstore.similarity_search("삼성전자가 만든 생성형 AI의 이름은?")


In [37]:
print(result_docs[0].page_content)

- 삼성전자가 자체 개발한 생성 AI 모델 '삼성 가우스'를 공개하였으며, 이 모델은 언어, 코드, 이미지의 3개 모델로 구성되어 온디바이스에서 작동 가능하다.  
- '삼성 가우스'는 정규분포 이론을 정립한 수학자 가우스의 이름을 따왔으며, 다양한 상황에 최적화된 모델 선택이 가능하다.  
- 삼성전자는 이 AI 모델이 사용자 정보를 외부로 유출하지 않도록 설계되었으며, 향후 다양한 제품에 단계적으로 탑재할 계획이다.


In [39]:
retrieved_docs = retriever.invoke("삼성전자가 만든 생성형 AI의 이름은?")
print(retrieved_docs[0].page_content)

SPRi AI Brief |  
2023-12월호
10
삼성전자, 자체 개발 생성 AI ‘삼성 가우스’ 공개
n 삼성전자가 온디바이스에서 작동 가능하며 언어, 코드, 이미지의 3개 모델로 구성된 자체 개발 생성 
AI 모델 ‘삼성 가우스’를 공개
n 삼성전자는 삼성 가우스를 다양한 제품에 단계적으로 탑재할 계획으로, 온디바이스 작동이 가능한 
삼성 가우스는 외부로 사용자 정보가 유출될 위험이 없다는 장점을 보유
KEY Contents
£ 언어, 코드, 이미지의 3개 모델로 구성된 삼성 가우스, 온디바이스 작동 지원
n 삼성전자가 2023년 11월 8일 열린 ‘삼성 AI 포럼 2023’ 행사에서 자체 개발한 생성 AI 모델 
‘삼성 가우스’를 최초 공개
∙정규분포 이론을 정립한 천재 수학자 가우스(Gauss)의 이름을 본뜬 삼성 가우스는 다양한 상황에 
최적화된 크기의 모델 선택이 가능
∙삼성 가우스는 라이선스나 개인정보를 침해하지 않는 안전한 데이터를 통해 학습되었으며, 
온디바이스에서 작동하도록 설계되어 외부로 사용자의 정보가 유출되지 않는 장점을 보유
∙삼성전자는 삼성 가우스를 활용한 온디바이스 AI 기술도 소개했으며, 생성 AI 모델을 다양한 제품에


## 가설 쿼리 문서 내용 탐색

In [40]:
functions = [
    {
        "name":"hypothetical_questions",
        "description":"Generate hypothetical questions",
        "parameters":{                # 함수의 매개변수를 정의
            "type": "object",         # 매개변수의 타입을 객체로 지정
            "properties":{            # 객체의 속성을 정의
                "questions":{         # question 속성을 정의
                    "type":"array",   # questions의 타입을 배열로 지정
                    "items":{
                        "type":"string"
                    },                # 배열의 요소 타입을 문자열로 지정
                },
            },
            "required":["questions"], # 필수 매개변수로 Questions를 지정
        },
    },
]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
from langchain_openai import ChatOpenAI

hypothetical_query_chain = (
    {"doc": lambda x: x.page_content}
    |ChatPromptTemplate.from_template(
        "Generate a list of exactly 3 hypothetical questions that the below document cloud be used to answer."
        "Potential users are those interested in the AI industry. Create questions that they would be interested in."
        "Output should be written in korean:\n\n{doc}"
    )
    |ChatOpenAI(max_retries=0, model="gpt-4o-mini").bind(
        functions=functions, function_call={"name":"hypothetical_questions"}
    )
    | JsonKeyOutputFunctionsParser(key_name="questions")
)

In [ ]:
hypothetical_query_chain.invoke(split_docs[33])

In [ ]:
hypothetical_questions = hypothetical_query_chain.batch(
    split_docs, {"max_concurrency": 10}
)